In [1]:
#Let's load everything
import pandas as pd
import numpy as np

rfm = pd.read_csv("../data/processed/rfm_features.csv")
campaign_design = pd.read_csv("../data/processed/campaign_design.csv")

segments = rfm.groupby("segment_name").agg(
    n_customers=("customer_unique_id", "count"),
    avg_order_value=("avg_order_value", "mean"),
).reset_index()

segments = segments.merge(campaign_design, on="segment_name")
segments

,segment_name,n_customers,avg_order_value,channel,offer_type,strategy
0,Casual Low-Spend Majority,46437,54.794206,Email,Free shipping threshold / small % off,Low-cost broad reach
1,Dissatisfied & Disengaged,13747,122.188100,Direct email,Service recovery credit,"Retention/recovery, not promotional"
2,High-Value Financed One-Timers,30950,267.642353,Email + retargeting,Free shipping / extended installments,Cross-sell into new categories
3,Loyal High-Value Repeaters,2224,148.276579,Personalized email,Loyalty tier / early access,"Recognition, deepen relationship"


In [2]:
#response rates
baseline_response_rate = 0.0016  # Klaviyo 2026 ecommerce placed-order rate

segment_multiplier_map = {
    "High-Value Financed One-Timers": 1.1,
    "Dissatisfied & Disengaged": 0.4,
    "Casual Low-Spend Majority": 0.8,
    "Loyal High-Value Repeaters": 2.5,
}

segments["segment_multiplier"] = segments["segment_name"].map(segment_multiplier_map)
segments["response_rate"] = baseline_response_rate * segments["segment_multiplier"]

In [3]:
#campaign cost per customer, tied to the channel each segment actually gets
# Document your cost assumptions — these should reflect real per-contact costs for each channel type
channel_cost_map = {
    "Email + retargeting": 0.20,      # email (~$0.01-0.05) + display retargeting media cost
    "Direct email": 0.05,             # plain transactional-style email, no ad spend
    "Email": 0.03,                    # simple bulk email, lowest cost
    "Personalized email": 0.15,       # more design/copy effort per segment, still email-only delivery
}

segments["campaign_cost_per_customer"] = segments["channel"].map(channel_cost_map)
segments[["segment_name", "channel", "campaign_cost_per_customer"]]

,segment_name,channel,campaign_cost_per_customer
0,Casual Low-Spend Majority,Email,0.03
1,Dissatisfied & Disengaged,Direct email,0.05
2,High-Value Financed One-Timers,Email + retargeting,0.20
3,Loyal High-Value Repeaters,Personalized email,0.15


In [4]:
#the core simulation math
segments["expected_responders"] = segments["n_customers"] * segments["response_rate"]
segments["expected_revenue"] = segments["expected_responders"] * segments["avg_order_value"]
segments["total_cost"] = segments["n_customers"] * segments["campaign_cost_per_customer"]
segments["net_revenue"] = segments["expected_revenue"] - segments["total_cost"]
segments["roi"] = segments["net_revenue"] / segments["total_cost"]

segments[["segment_name", "n_customers", "response_rate", "expected_responders",
          "expected_revenue", "total_cost", "net_revenue", "roi"]].round(2)

,segment_name,n_customers,response_rate,expected_responders,expected_revenue,total_cost,net_revenue,roi
0,Casual Low-Spend Majority,46437,0.0,59.44,3256.93,1393.11,1863.82,1.34
1,Dissatisfied & Disengaged,13747,0.0,8.80,1075.02,687.35,387.67,0.56
2,High-Value Financed One-Timers,30950,0.0,54.47,14579.01,6190.00,8389.01,1.36
3,Loyal High-Value Repeaters,2224,0.0,8.90,1319.07,333.60,985.47,2.95


In [5]:
# totals
segmented_total_cost = segments["total_cost"].sum()
segmented_total_revenue = segments["expected_revenue"].sum()
segmented_net_revenue = segments["net_revenue"].sum()
segmented_roi = segmented_net_revenue / segmented_total_cost

print(f"Segmented campaign — Total cost: ${segmented_total_cost:,.2f}")
print(f"Segmented campaign — Expected revenue: ${segmented_total_revenue:,.2f}")
print(f"Segmented campaign — Net revenue: ${segmented_net_revenue:,.2f}")
print(f"Segmented campaign — ROI: {segmented_roi:.2f}x")

Segmented campaign — Total cost: $8,604.06
Segmented campaign — Expected revenue: $20,230.04
Segmented campaign — Net revenue: $11,625.98
Segmented campaign — ROI: 1.35x


In [6]:
segments.to_csv("../data/processed/simulation_results.csv", index=False)

In [7]:
# the spray-and-pray baseline. One campaign, one channel, one offer, applied to everyone at a blended (non-segmented) rate:
total_customers = segments["n_customers"].sum()

# Spray-and-pray uses a single blended channel cost and the UNMULTIPLIED baseline response rate —
# no segment-specific targeting means no segment-specific lift
spray_cost_per_customer = segments["campaign_cost_per_customer"].mean()  # blended average cost across channels
spray_response_rate = baseline_response_rate  # the raw Klaviyo benchmark, no multiplier applied
spray_avg_order_value = rfm["avg_order_value"].mean()  # blended AOV across the whole base, not segment-specific

spray_total_cost = total_customers * spray_cost_per_customer
spray_responders = total_customers * spray_response_rate
spray_revenue = spray_responders * spray_avg_order_value
spray_net_revenue = spray_revenue - spray_total_cost
spray_roi = spray_net_revenue / spray_total_cost

print(f"Spray-and-pray — Total cost: ${spray_total_cost:,.2f}")
print(f"Spray-and-pray — Expected revenue: ${spray_revenue:,.2f}")
print(f"Spray-and-pray — Net revenue: ${spray_net_revenue:,.2f}")
print(f"Spray-and-pray — ROI: {spray_roi:.2f}x")

Spray-and-pray — Total cost: $10,035.99
Spray-and-pray — Expected revenue: $20,539.99
Spray-and-pray — Net revenue: $10,504.01
Spray-and-pray — ROI: 1.05x


In [8]:
# the comparison, and your headline number:
lift_pct = (segmented_net_revenue - spray_net_revenue) / abs(spray_net_revenue) * 100
cost_diff_pct = (segmented_total_cost - spray_total_cost) / spray_total_cost * 100

print(f"Segmented net revenue: ${segmented_net_revenue:,.2f}")
print(f"Spray-and-pray net revenue: ${spray_net_revenue:,.2f}")
print(f"Net revenue lift from segmentation: {lift_pct:.1f}%")
print(f"Cost difference (segmented vs spray): {cost_diff_pct:+.1f}%")

Segmented net revenue: $11,625.98
Spray-and-pray net revenue: $10,504.01
Net revenue lift from segmentation: 10.7%
Cost difference (segmented vs spray): -14.3%


In [ ]:
# the headline chart:
import matplotlib.pyplot as plt

comparison = pd.DataFrame({
    "Approach": ["Spray-and-Pray", "Segmented"],
    "Total Cost": [spray_total_cost, segmented_total_cost],
    "Expected Revenue": [spray_revenue, segmented_total_revenue],
    "Net Revenue": [spray_net_revenue, segmented_net_revenue],
})

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison))
width = 0.25

ax.bar(x - width, comparison["Total Cost"], width, label="Total Cost", color="lightcoral")
ax.bar(x, comparison["Expected Revenue"], width, label="Expected Revenue", color="steelblue")
ax.bar(x + width, comparison["Net Revenue"], width, label="Net Revenue", color="seagreen")

ax.set_xticks(x)
ax.set_xticklabels(comparison["Approach"])
ax.set_ylabel("USD")
ax.set_title(f"Spray-and-Pray vs Segmented Campaign: {lift_pct:.0f}% Net Revenue Lift")
ax.legend()
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig("../outputs/headline_comparison.png", dpi=150)
plt.show()